# 🚀 01. 데이터베이스 연동 및 기본 임베딩 실습

이 노트북은 프로젝트의 설정 정보를 불러오고, PostgreSQL 데이터베이스 연결 상태를 점검하며, OpenAI 임베딩 API를 활용해 3072차원 벡터를 생성하고, pgvector 유사도 검색을 실습하는 공간입니다.

### ⚙️ 필요한 라이브러리 및 모듈 임포트
프로젝트의 공통 설정 모듈(`api.common.config.settings`)과 SQLAlchemy 비동기 연결 객체, LangChain 임베딩 모듈들을 불러옵니다.

In [ ]:
import asyncio
from sqlalchemy import text
from api.common.config import settings
from api.database.config.dbsession import engine
from langchain.embeddings import init_embeddings
from langchain_postgres import PGVector

## 📌 1단계. 데이터베이스 연결 상태(Ping) 테스트
SQLAlchemy 비동기 엔진을 사용하여 PostgreSQL DB와 정상 연결되는지 `SELECT 1` 쿼리로 확인합니다. 또한, pgvector 익스텐션(`vector`)이 활성화되어 있는지도 검증합니다.

In [ ]:
async def check_database_connection():
    print(f"Database URL: {settings.DATABASE_URL}")
    async with engine.connect() as conn:
        result = await conn.execute(text("SELECT 1"))
        print(f"SELECT 1 returned: {result.scalar()}")
        
        # pgvector 익스텐션 활성화 확인
        vector_ext = await conn.execute(text("SELECT extname FROM pg_extension WHERE extname = 'vector';"))
        ext = vector_ext.scalar()
        if ext:
            print("PostgreSQL pgvector extension is ACTIVE! 🎉")
        else:
            print("WARNING: pgvector extension is NOT active. Ensure table DDL commands have run.")

await check_database_connection()

## 📌 2단계. OpenAI text-embedding-3-large 임베딩 생성
LangChain의 `init_embeddings`를 사용하여 OpenAI `text-embedding-3-large` (3072차원) 모델 인스턴스를 초기화하고, 임시 문장에 대한 임베딩 벡터 생성을 진행합니다.

In [ ]:
embeddings = init_embeddings(model="openai:text-embedding-3-large")
sample_text = "Graph neural networks and attention mechanisms for paper recommendation."

print(f"Generating embedding for: '{sample_text}'")
vector = await embeddings.aembed_query(sample_text)
print("Embedding generation success!")
print(f"Vector Dimensions: {len(vector)} (Expected: 3072)")
print(f"Vector Snippet (First 5 values): {vector[:5]}")

## 📌 3단계. pgvector 유사도 검색 실습
데이터를 pgvector 스토어에 적재하고, 코사인 유사도 연산 기반으로 질문과 가장 가까운 문서를 찾아 반환하는 RAG 유사도 검색의 기초를 실습합니다.

In [ ]:
collection_name = "demo_notebook_embeddings"
vectorstore = PGVector(
    embeddings=embeddings,
    collection_name=collection_name,
    connection=settings.PGVECTOR_URL,
    async_mode=True,
)

demo_texts = [
    "We propose a new model called BERT for natural language processing.",
    "Attention mechanisms and transformers are widely used in modern language models.",
    "PostgreSQL is an open-source relational database with pgvector extension.",
    "Biotechnology research on CRISPR and gene sequencing advances rapidly."
]

print(f"Adding {len(demo_texts)} demo texts to collection '{collection_name}'...")
metadatas = [{"source": "demo", "doc_idx": i} for i in range(len(demo_texts))]
await vectorstore.aadd_texts(demo_texts, metadatas=metadatas)
print("Successfully ingested demo documents!")

In [ ]:
query = "What database is used for vector search?"
print(f"Searching for query: '{query}'")

results = await vectorstore.asimilarity_search_with_score(query, k=2)
for idx, (doc, score) in enumerate(results, 1):
    similarity = round(1.0 - score, 4)
    print(f"  [{idx}] Text: {doc.page_content}")
    print(f"      Similarity Score: {similarity:.4f} (Distance: {score:.4f})")
    print(f"      Metadata: {doc.metadata}")

In [ ]:
# 학습 완료 후 데모용 컬렉션 공간 정리
print("Cleaning up demo collection...")
await vectorstore.adelete_collection()
print("Demo collection deleted successfully! 🎉")